### Implementation of Data Encryption Standard (DES)

This notebook presents a step-by-step implementation of the Data Encryption Standard (DES) algorithm from scratch.

DES is a symmetric-key encryption algorithm that operates on 64-bit blocks of data using a 56-bit key. It follows a Feistel structure and performs 16 rounds of processing involving permutation, substitution, and key mixing.

The goal of this notebook is to demonstrate the working of DES in a clear and modular manner without using any external cryptographic libraries.

---

### Objectives
- To understand the internal structure of DES  
- To implement DES using basic programming constructs  
- To observe how plaintext is transformed into ciphertext through multiple rounds

In [1]:
def permute(block, table):
    return ''.join(block[i-1] for i in table)
def xor(a, b):
    return ''.join('0' if i==j else '1' for i,j in zip(a, b))

### Helper Functions

The following helper functions are used throughout the DES implementation:

- permute(block, table):  
  Rearranges the bits of the input block according to a given permutation table.  
  This is used in multiple stages such as initial permutation, expansion, and final permutation.

- xor(a, b):  
  Performs a bitwise XOR operation between two binary strings of equal length.  
  XOR is a key operation in DES for mixing data with the key.

In [2]:
print(permute("1010", [3,1,4,2]))
print(xor("1100", "1010"))

1100
0110


### DES Tables

DES uses predefined permutation and substitution tables.  
These tables are part of the standard specification and are used in different stages of the algorithm such as initial permutation, expansion, substitution (S-boxes), and final permutation.

The following tables are used in this implementation.

Initial Permutation (IP) - Shuffles the 64-bit input before the processing starts
Final Permutation (FP) - Shuffles the bits after the 16 rounds have been completed.

In [3]:
# Initial Permutation Table (IP)
IP = [
58, 50, 42, 34, 26, 18, 10, 2,
60, 52, 44, 36, 28, 20, 12, 4,
62, 54, 46, 38, 30, 22, 14, 6,
64, 56, 48, 40, 32, 24, 16, 8,
57, 49, 41, 33, 25, 17, 9, 1,
59, 51, 43, 35, 27, 19, 11, 3,
61, 53, 45, 37, 29, 21, 13, 5,
63, 55, 47, 39, 31, 23, 15, 7
]

# Final Permutation Table (FP)
FP = [
40, 8, 48, 16, 56, 24, 64, 32,
39, 7, 47, 15, 55, 23, 63, 31,
38, 6, 46, 14, 54, 22, 62, 30,
37, 5, 45, 13, 53, 21, 61, 29,
36, 4, 44, 12, 52, 20, 60, 28,
35, 3, 43, 11, 51, 19, 59, 27,
34, 2, 42, 10, 50, 18, 58, 26,
33, 1, 41, 9, 49, 17, 57, 25
]

In [4]:
test = "0" * 64
print(len(permute(test, IP)))  # should be 64

64


### Expansion, Substitution, and Permutation

In each DES round, the right half of the data is transformed using three main steps:

1. Expansion: Expands 32 bits to 48 bits  
2. Substitution: Reduces 48 bits to 32 bits using S-boxes  
3. Permutation: Rearranges the bits  

These operations together provide confusion and diffusion in the DES algorithm.


In [5]:
E = [
32, 1, 2, 3, 4, 5,
4, 5, 6, 7, 8, 9,
8, 9,10,11,12,13,
12,13,14,15,16,17,
16,17,18,19,20,21,
20,21,22,23,24,25,
24,25,26,27,28,29,
28,29,30,31,32,1
]

In [6]:
right = "0" * 32
expanded = permute(right, E)
print(len(expanded))  # should be 48

48


In [7]:
S1 = [
[14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7],
[0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8],
[4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0],
[15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]
]

In [8]:
S_BOXES = [S1,
[
[15,1,8,14,6,11,3,4,9,7,2,13,12,0,5,10],
[3,13,4,7,15,2,8,14,12,0,1,10,6,9,11,5],
[0,14,7,11,10,4,13,1,5,8,12,6,9,3,2,15],
[13,8,10,1,3,15,4,2,11,6,7,12,0,5,14,9]
],
[
[10,0,9,14,6,3,15,5,1,13,12,7,11,4,2,8],
[13,7,0,9,3,4,6,10,2,8,5,14,12,11,15,1],
[13,6,4,9,8,15,3,0,11,1,2,12,5,10,14,7],
[1,10,13,0,6,9,8,7,4,15,14,3,11,5,2,12]
],
[
[7,13,14,3,0,6,9,10,1,2,8,5,11,12,4,15],
[13,8,11,5,6,15,0,3,4,7,2,12,1,10,14,9],
[10,6,9,0,12,11,7,13,15,1,3,14,5,2,8,4],
[3,15,0,6,10,1,13,8,9,4,5,11,12,7,2,14]
],
[
[2,12,4,1,7,10,11,6,8,5,3,15,13,0,14,9],
[14,11,2,12,4,7,13,1,5,0,15,10,3,9,8,6],
[4,2,1,11,10,13,7,8,15,9,12,5,6,3,0,14],
[11,8,12,7,1,14,2,13,6,15,0,9,10,4,5,3]
],
[
[12,1,10,15,9,2,6,8,0,13,3,4,14,7,5,11],
[10,15,4,2,7,12,9,5,6,1,13,14,0,11,3,8],
[9,14,15,5,2,8,12,3,7,0,4,10,1,13,11,6],
[4,3,2,12,9,5,15,10,11,14,1,7,6,0,8,13]
],
[
[4,11,2,14,15,0,8,13,3,12,9,7,5,10,6,1],
[13,0,11,7,4,9,1,10,14,3,5,12,2,15,8,6],
[1,4,11,13,12,3,7,14,10,15,6,8,0,5,9,2],
[6,11,13,8,1,4,10,7,9,5,0,15,14,2,3,12]
],
[
[13,2,8,4,6,15,11,1,10,9,3,14,5,0,12,7],
[1,15,13,8,10,3,7,4,12,5,6,11,0,14,9,2],
[7,11,4,1,9,12,14,2,0,6,10,13,15,3,5,8],
[2,1,14,7,4,10,8,13,15,12,9,0,3,5,6,11]
]]

In [9]:
P = [
16, 7, 20, 21,
29,12,28,17,
1,15,23,26,
5,18,31,10,
2, 8, 24,14,
32,27, 3, 9,
19,13,30, 6,
22,11, 4,25
]

### S-Box Lookup

The S-box reduces 6 bits into 4 bits.

- The first and last bits determine the row  
- The middle four bits determine the column  
- The value from the S-box is converted back to a 4-bit binary number  

This step introduces non-linearity into DES, making it more secure.

In [10]:
def sbox_lookup(sbox, block6):
    # row = first and last bit
    row = int(block6[0] + block6[5], 2)
    
    # column = middle 4 bits
    col = int(block6[1:5], 2)
    
    value = sbox[row][col]
    
    # convert to 4-bit binary
    return format(value, '04b')

In [11]:
print(sbox_lookup(S1, "101011"))

1001


### Feistel Function

The Feistel function is the core transformation in each DES round.

It performs the following steps:
1. Expansion of 32-bit input to 48 bits  
2. XOR with round key  
3. Substitution using 8 S-boxes  
4. Permutation of the result  

This function is applied to the right half of the data in each round.

In [12]:
def feistel(right, key):
    # Step 1: Expansion
    expanded = permute(right, E)
    
    # Step 2: XOR with key
    xored = xor(expanded, key)
    
    # Step 3: Split into 8 blocks of 6 bits
    blocks = [xored[i:i+6] for i in range(0, 48, 6)]
    
    # Step 4: Apply S-boxes
    result = ''
    for i in range(8):
        result += sbox_lookup(S_BOXES[i], blocks[i])
    
    # Step 5: Permutation
    result = permute(result, P)
    
    return result

In [13]:
print(len(feistel("0"*32, "0"*48)))  # should be 32

32


### Key Generation

DES uses a 56-bit key (originally provided as 64 bits, where 8 bits are ignored) to generate 16 different round keys. Each round key is 48 bits long and is used in one round of the DES algorithm.

The key generation process involves the following steps:

1. Permuted Choice 1 (PC-1):  
   The original 64-bit key is permuted and reduced to 56 bits by discarding every 8th bit.

2. Splitting:  
   The 56-bit key is divided into two halves:
   - Left half (28 bits)
   - Right half (28 bits)

3. Left Shifts:  
   In each of the 16 rounds, both halves are shifted left by 1 or 2 bits.  
   The number of shifts depends on the round number.

4. Permuted Choice 2 (PC-2):  
   The shifted halves are combined and compressed into a 48-bit round key using another permutation.

This process is repeated 16 times to generate 16 unique round keys, which are used in each round of the DES encryption process.

In [14]:
# Permuted Choice 1 (PC-1)
PC1 = [
57,49,41,33,25,17,9,
1,58,50,42,34,26,18,
10,2,59,51,43,35,27,
19,11,3,60,52,44,36,
63,55,47,39,31,23,15,
7,62,54,46,38,30,22,
14,6,61,53,45,37,29,
21,13,5,28,20,12,4
]

# Permuted Choice 2 (PC-2)
PC2 = [
14,17,11,24,1,5,
3,28,15,6,21,10,
23,19,12,4,26,8,
16,7,27,20,13,2,
41,52,31,37,47,55,
30,40,51,45,33,48,
44,49,39,56,34,53,
46,42,50,36,29,32
]

# Left shifts for each round
SHIFT_TABLE = [
1, 1, 2, 2,
2, 2, 2, 2,
1, 2, 2, 2,
2, 2, 2, 1
]

In [15]:
def generate_keys(key):
    # Step 1: Apply PC1
    key = permute(key, PC1)
    
    # Step 2: Split into left and right (28 bits each)
    left = key[:28]
    right = key[28:]
    
    keys = []
    
    # Step 3: Generate 16 keys
    for shift in SHIFT_TABLE:
        # Left shift
        left = left[shift:] + left[:shift]
        right = right[shift:] + right[:shift]
        
        # Combine
        combined = left + right
        
        # Apply PC2 → 48-bit key
        round_key = permute(combined, PC2)
        
        keys.append(round_key)
    
    return keys

In [16]:
keys = generate_keys("0"*64)
print(len(keys))        # should be 16
print(len(keys[0]))     # should be 48

16
48


### DES Encryption

The DES encryption process combines all previously defined components.

Steps involved:
1. Apply the initial permutation to the plaintext  
2. Split the block into left and right halves  
3. Perform 16 rounds of Feistel operations using round keys  
4. Swap the final halves  
5. Apply the final permutation to produce ciphertext  

This function completes the DES encryption process.

In [17]:
def des_encrypt(plaintext, key):
    # Step 1: Initial Permutation
    block = permute(plaintext, IP)
    
    # Step 2: Split into left and right
    left = block[:32]
    right = block[32:]
    
    # Step 3: Generate keys
    keys = generate_keys(key)
    
    # Step 4: 16 rounds
    for i in range(16):
        new_left = right
        new_right = xor(left, feistel(right, keys[i]))
        left, right = new_left, new_right
    
    # Step 5: Swap halves
    combined = right + left
    
    # Step 6: Final Permutation
    ciphertext = permute(combined, FP)
    
    return ciphertext

In [18]:
plaintext = "0" * 64
key = "0" * 64

cipher = des_encrypt(plaintext, key)
print(cipher)
print(len(cipher))  # should be 64

1000110010100110010011011110100111000001101100010010001110100111
64


### Test Cases

The following test cases are used to validate the correctness and behavior of the DES implementation. Each case demonstrates different input patterns and how the algorithm transforms them.

---

Test Case 1: All Zeros  
Plaintext and key contain all zeros. This verifies the basic functionality of the algorithm.

Test Case 2: All Ones  
Plaintext and key contain all ones. This tests the algorithm under extreme input conditions.

Test Case 3: Alternating Bits  
Plaintext and key use alternating bit patterns to observe how DES removes visible patterns.

Test Case 4: Half Zeros, Half Ones  
The input is split into two halves to observe how the Feistel structure mixes data.

Test Case 5: Random Pattern  
A random-looking binary input is used to simulate realistic encryption scenarios.

Test Case 6: Single Bit Change (Avalanche Effect)  
Two plaintexts differing by one bit are used to demonstrate the avalanche effect, where a small input change leads to a large output change.

In [19]:
# Test Case 1
plaintext = "0" * 64
key = "0" * 64
print("TC1:", des_encrypt(plaintext, key))

# Test Case 2
plaintext = "1" * 64
key = "1" * 64
print("TC2:", des_encrypt(plaintext, key))

# Test Case 3
plaintext = "10101010" * 8
key = "01010101" * 8
print("TC3:", des_encrypt(plaintext, key))

# Test Case 4
plaintext = "0" * 32 + "1" * 32
key = "1" * 32 + "0" * 32
print("TC4:", des_encrypt(plaintext, key))

# Test Case 5
plaintext = "1100101011110000101010101111000010101010111100001010101011110000"
key       = "1010101111001101111001101010101111001101111001101010101111001101"
print("TC5:", des_encrypt(plaintext, key))

# Test Case 6 (Avalanche Effect)
plaintext1 = "0" * 64
plaintext2 = "0" * 63 + "1"
key = "0" * 64

print("TC6a:", des_encrypt(plaintext1, key))
print("TC6b:", des_encrypt(plaintext2, key))

TC1: 1000110010100110010011011110100111000001101100010010001110100111
TC2: 0111001101011001101100100001011000111110010011101101110001011000
TC3: 0011010000111010000010011111100110110010110010110101110011001010
TC4: 1000001101000100101001111111011001000001100010000010111110001111
TC5: 1010101110110101100001111111100110101110011110010000101001011001
TC6a: 1000110010100110010011011110100111000001101100010010001110100111
TC6b: 0001011001101011010000001011010001001010101110100100101111010110
